# Medical LLM Continued Pretraining

This notebook performs domain-adaptive continued pretraining on Qwen2.5-7B-Instruct using a medical corpus.

**Features:**
- Automatically detects and loads local model, or downloads from HuggingFace if missing
- Freezes all parameters except last 8 transformer layers + LM head + LayerNorm
- BF16 precision training with gradient checkpointing
- Early stopping with patience=3
- CSV logging of training metrics
- Trains on 2,637 medical chunks from 13 books (eval: 16 chunks from held-out book)

## Cell 1: System Check & Imports

In [ ]:
import os
import sys
import json
import torch
import warnings
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')

# Verify GPU
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

# Verify disk space
import shutil
total, used, free = shutil.disk_usage("/" if os.name != 'nt' else "C:\\")
print(f"Disk space available: {free / 1e9:.1f} GB")

if free / 1e9 < 50:
    print("WARNING: Less than 50GB free. Model download/training may fail.")

## Cell 2: Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT_DIR = Path.cwd()
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MODEL_LOCAL_PATH = ROOT_DIR / "Qwen2.5-7B-Instruct"
TRAIN_DATA = ROOT_DIR / "data" / "final" / "train.jsonl"
EVAL_DATA = ROOT_DIR / "data" / "final" / "eval.jsonl"
OUTPUT_DIR = ROOT_DIR / "medical_qwen_cpt"
LOGS_DIR = ROOT_DIR / "logs"

LOGS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Training Hyperparameters ──────────────────────────────────────────────────
LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
EFFECTIVE_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
MAX_SEQ_LENGTH = 1024
EVAL_STEPS = 100
SAVE_STEPS = 100
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 3

# ── Precision ─────────────────────────────────────────────────────────────────
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP32 = not USE_BF16

print(f"\n{'─'*70}")
print("Configuration Summary")
print(f"{'─'*70}")
print(f"Model: {MODEL_NAME}")
print(f"Local path: {MODEL_LOCAL_PATH}")
print(f"Train data: {TRAIN_DATA} ({len(open(TRAIN_DATA).readlines())} chunks)")
print(f"Eval data: {EVAL_DATA} ({len(open(EVAL_DATA).readlines())} chunks)")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Batch size (effective): {EFFECTIVE_BATCH_SIZE}")
print(f"Epochs: {NUM_TRAIN_EPOCHS}")
print(f"Precision: {'BF16' if USE_BF16 else 'FP32'}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"{'─'*70}\n")

## Cell 3: Model Loading (Auto-Download or Local)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import subprocess

print("Checking for model...")

if MODEL_LOCAL_PATH.exists():
    print(f"✓ Found local model at {MODEL_LOCAL_PATH}")
    tokenizer = AutoTokenizer.from_pretrained(str(MODEL_LOCAL_PATH))
    print("✓ Tokenizer loaded")
else:
    print(f"Model not found at {MODEL_LOCAL_PATH}")
    print(f"Downloading {MODEL_NAME} from HuggingFace...")
    print("(This may take 10-20 minutes and requires ~30GB disk space)\n")
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
        print(f"✓ Tokenizer loaded from HuggingFace")
        
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            trust_remote_code=True,
            torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
            device_map="auto"
        )
        print(f"✓ Model downloaded and loaded")
        
        print(f"\nSaving to {MODEL_LOCAL_PATH}...")
        model.save_pretrained(str(MODEL_LOCAL_PATH))
        tokenizer.save_pretrained(str(MODEL_LOCAL_PATH))
        print(f"✓ Model saved locally")
        
    except Exception as e:
        print(f"❌ Download failed: {e}")
        print("Please check your internet connection and disk space.")
        raise

print(f"\nTokenizer vocab size: {len(tokenizer)}")

## Cell 4: Load Model for Training

In [ ]:
print(f"Loading model from {MODEL_LOCAL_PATH}...")

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_LOCAL_PATH),
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
    device_map="auto"
)

model.gradient_checkpointing_enable()
print("✓ Model loaded with gradient checkpointing")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Cell 5: Freeze/Unfreeze Strategy

In [ ]:
# Freeze all parameters
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 8 transformer layers
num_layers = len(model.model.layers)
for layer in model.model.layers[-8:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze LM head and layer norm
for param in model.lm_head.parameters():
    param.requires_grad = True

for param in model.model.norm.parameters():
    param.requires_grad = True

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_pct = (trainable_params / total_params) * 100

print(f"\n{'─'*70}")
print("Parameter Freeze Strategy")
print(f"{'─'*70}")
print(f"Total parameters: {total_params/1e9:.2f}B")
print(f"Trainable parameters: {trainable_params/1e9:.2f}B ({trainable_pct:.2f}%)")
print(f"Frozen parameters: {(total_params - trainable_params)/1e9:.2f}B")
print(f"\nUnfrozen components:")
print(f"  - Last 8 transformer layers")
print(f"  - LM head")
print(f"  - Layer normalization")
print(f"{'─'*70}\n")

## Cell 6: Load Training Data

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

print("Loading training data...")
train_data = load_jsonl(TRAIN_DATA)
eval_data = load_jsonl(EVAL_DATA)

print(f"✓ Train: {len(train_data)} chunks")
print(f"✓ Eval: {len(eval_data)} chunks")

# Create datasets
train_dataset = Dataset.from_dict({"text": [d["text"] for d in train_data]})
eval_dataset = Dataset.from_dict({"text": [d["text"] for d in eval_data]})

print(f"\nSample training chunk (first 200 chars):")
print(f"'{train_data[0]['text'][:200]}...'")

## Cell 7: Tokenization

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False
    )

print("Tokenizing datasets...")
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=["text"]
)

eval_tokenized = eval_dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=["text"]
)

print(f"✓ Tokenization complete")
print(f"  Train dataset: {len(train_tokenized)} chunks")
print(f"  Eval dataset: {len(eval_tokenized)} chunks")

## Cell 8: Custom Callbacks

In [ ]:
from transformers import TrainerCallback
import csv

class TrainingCSVCallback(TrainerCallback):
    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.csv_file = open(csv_path, 'w', newline='')
        self.writer = csv.DictWriter(self.csv_file, fieldnames=['step', 'train_loss', 'eval_loss', 'eval_perplexity'])
        self.writer.writeheader()
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            self.writer.writerow({
                'step': state.global_step,
                'train_loss': logs.get('loss', ''),
                'eval_loss': logs.get('eval_loss', ''),
                'eval_perplexity': logs.get('eval_perplexity', '')
            })
            self.csv_file.flush()
    
    def __del__(self):
        if self.csv_file and not self.csv_file.closed:
            self.csv_file.close()

class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=3):
        self.patience = patience
        self.patience_counter = 0
        self.best_eval_loss = float('inf')
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        eval_loss = metrics.get('eval_loss', float('inf'))
        if eval_loss < self.best_eval_loss:
            self.best_eval_loss = eval_loss
            self.patience_counter = 0
        else:
            self.patience_counter += 1
            if self.patience_counter >= self.patience:
                print(f"Early stopping triggered (patience={self.patience})")
                control.should_training_stop = True

print("✓ Custom callbacks defined")

## Cell 9: Training Setup

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

csv_log_path = LOGS_DIR / "training_metrics.csv"

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    logging_strategy="steps",
    logging_steps=10,
    logging_dir=str(LOGS_DIR),
    bf16=USE_BF16,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    seed=42,
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Training Arguments:")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size (effective): {PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Evaluation steps: {training_args.eval_steps}")
print(f"  Total epochs: {training_args.num_train_epochs}")
print(f"  Precision: {'BF16' if training_args.bf16 else 'FP32'}")

## Cell 10: Create Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
    callbacks=[
        TrainingCSVCallback(str(csv_log_path)),
        EarlyStoppingCallback(patience=EARLY_STOPPING_PATIENCE)
    ]
)

print("✓ Trainer created and ready")
print(f"\nTraining will start with:")
print(f"  {len(train_tokenized)} training chunks")
print(f"  {len(eval_tokenized)} evaluation chunks")
print(f"  {EARLY_STOPPING_PATIENCE} patience for early stopping")
print(f"\nMetrics will be logged to: {csv_log_path}")

## Cell 11: Train!

In [ ]:
import time

print(f"\n{'═'*70}")
print("Starting Medical LLM Continued Pretraining")
print(f"{'═'*70}")
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'═'*70}\n")

start_time = time.time()

train_result = trainer.train()

elapsed_time = time.time() - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

print(f"\n{'═'*70}")
print("Training Complete!")
print(f"{'═'*70}")
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total training time: {hours}h {minutes}m {seconds}s")
print(f"Final training loss: {train_result.training_loss:.4f}")
print(f"{'═'*70}\n")

## Cell 12: Save Model

In [ ]:
print(f"Saving model to {OUTPUT_DIR}...")

model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

print(f"✓ Model saved")
print(f"✓ Tokenizer saved")

# Save training config
config = {
    "base_model": MODEL_NAME,
    "training_data_chunks": len(train_data),
    "eval_data_chunks": len(eval_data),
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_TRAIN_EPOCHS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "trainable_parameters": trainable_params,
    "total_parameters": total_params,
    "final_loss": float(train_result.training_loss),
    "training_time_seconds": elapsed_time
}

config_path = OUTPUT_DIR / "training_config.json"
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✓ Training config saved to {config_path}")

## Cell 13: Visualize Metrics

In [ ]:
# Load training metrics from CSV
if csv_log_path.exists():
    df = pd.read_csv(csv_log_path)
    df = df.dropna()
    
    if len(df) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Training loss
        axes[0].plot(df['step'], df['train_loss'], 'b-o', label='Training Loss')
        axes[0].plot(df['step'], df['eval_loss'], 'r-s', label='Eval Loss')
        axes[0].set_xlabel('Step')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training & Evaluation Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Perplexity
        axes[1].plot(df['step'], df['eval_perplexity'], 'g-o')
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Perplexity')
        axes[1].set_title('Evaluation Perplexity')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plot_path = LOGS_DIR / "training_curves.png"
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        print(f"✓ Metrics plot saved to {plot_path}")
        plt.show()
        
        print(f"\nTraining Summary:")
        print(f"  Final train loss: {df['train_loss'].iloc[-1]:.4f}")
        print(f"  Final eval loss: {df['eval_loss'].iloc[-1]:.4f}")
        print(f"  Final perplexity: {df['eval_perplexity'].iloc[-1]:.2f}")
    else:
        print("No metrics recorded (training may have been too short)")
else:
    print(f"Metrics file not found at {csv_log_path}")

## Cell 14: Summary & Next Steps

In [ ]:
print(f"\n{'═'*70}")
print("Medical LLM Continued Pretraining Complete")
print(f"{'═'*70}")
print(f"\nModel Location: {OUTPUT_DIR}")
print(f"\nTo use the trained model:")
print(f"  from transformers import AutoModelForCausalLM, AutoTokenizer")
print(f"  model = AutoModelForCausalLM.from_pretrained('{OUTPUT_DIR}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{OUTPUT_DIR}')")
print(f"\nTraining Artifacts:")
print(f"  - Model: {OUTPUT_DIR / 'pytorch_model.bin'}")
print(f"  - Tokenizer: {OUTPUT_DIR / 'tokenizer.json'}")
print(f"  - Config: {OUTPUT_DIR / 'config.json'}")
print(f"  - Training logs: {LOGS_DIR}")
print(f"\n{'═'*70}")